# PrimeKG Implementation
Will use `Llama-3.1-8B-Instruct`

In [3]:
import kuzu
import pandas as pd

In [5]:
kg = pd.read_csv("data/raw/kg.csv", low_memory=False)

x = kg[["x_index", "x_name", "x_type"]].rename(
    columns={"x_index": "node_index", "x_name": "name", "x_type": "type"})
y = kg[["y_index", "y_name", "y_type"]].rename(
    columns={"y_index": "node_index", "y_name": "name", "y_type": "type"})

entities = pd.concat([x, y]).drop_duplicates(subset="node_index").reset_index(drop=True)
entities.to_csv("data/processed/entities.csv", index=False)
print(f"{len(entities)} unique entities")

129375 unique entities


In [6]:
edges = kg[["x_index", "y_index", "relation", "display_relation"]].rename(
    columns={"x_index": "src", "y_index": "dst"})
edges.to_csv("data/processed/edges.csv", index=False)
print(f"{len(edges)} edges")

8100498 edges


In [ ]:
db = kuzu.Database("../primekg_kuzu")
conn = kuzu.Connection(db)

conn.execute("""
    CREATE NODE TABLE Entity(
        node_index INT64,
        name STRING,
        type STRING,
        PRIMARY KEY(node_index)
    )
""")

conn.execute("""
    CREATE REL TABLE RELATES_TO(
        FROM Entity TO Entity,
        relation STRING,
        display_relation STRING
    )
""")
print("Schema created")

In [ ]:
conn.execute("COPY Entity FROM '../data/processed/entities.csv' (HEADER=true)")
conn.execute("COPY RELATES_TO FROM '../data/processed/edges.csv' (HEADER=true)")
print("Data loaded")

In [ ]:
conn.execute("CALL CREATE_FTS_INDEX('Entity', 'entity_fts', ['name'])")
print("FTS index built")

In [ ]:
result = conn.execute("""
    CALL QUERY_FTS_INDEX('Entity', 'entity_fts', 'Alzheimer')
    YIELD node, score
    RETURN node.name, score
    ORDER BY score DESC
    LIMIT 5
""")
while result.has_next():
    print(result.get_next())